# 02: monitoring the training and diagnosing its failures (milestone m2)

Same data and same dense baseline as `01_dense_baseline.ipynb` (milestone m1). Here we follow Session 6 and TP06: we
plot the training and validation curves, read them, check that the training itself works, and measure what each tool
against overfitting changes.

## Imports

- `torch`, `torch.nn`: the networks; `DataLoader` and `TensorDataset` serve mini-batches from tensors.
- `torchvision.datasets`: `ImageFolder` reads the subset (one folder per genre).
- `copy`: `copy.deepcopy` makes an independent copy of the weights (for early stopping).
- `pandas`: reads `metadata.csv` (the artist of each image) and holds the comparison table.
- `numpy`, `matplotlib`: arrays and training curves.

In [1]:
import os
import copy
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

DATA_DIR = "data/wikiart_genre5"
np.random.seed(0)
torch.manual_seed(0)
torch.set_num_threads(2)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## Helper functions

The training tools of TP06:

- `train_one_epoch(model, loader, loss_fn, optimizer)`: one pass over the training data with the 5 lines of the
  training loop; returns the mean training loss.
- `evaluate(model, loader, loss_fn)`: returns `(mean loss, accuracy)` in evaluation mode, without gradients.
- `fit(model, optimizer, n_epochs, ...)`: trains for `n_epochs` and returns a **history**, a dictionary of lists with
  one value per epoch: `"train_loss"`, `"train_acc"`, `"val_loss"`, `"val_acc"`, `"lr"`. The train metrics are
  measured in evaluation mode on the whole training set after each epoch (`record`), so that train and validation
  are measured the same way.
- `plot_history(history)` and `plot_compare(histories, key)`: the curves.

In [2]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(yb)
        n += len(yb)
    return total / n


def evaluate(model, loader, loss_fn):
    model.eval()
    total, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            total += loss_fn(out, yb).item() * len(yb)
            correct += (out.argmax(dim=1) == yb).sum().item()
            n += len(yb)
    return total / n, correct / n


def new_history():
    return {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}


def record(history, model, optimizer):
    """Measure the model on the training and validation sets and append the results to the history."""
    tl, ta = evaluate(model, train_eval_loader, loss_fn)
    vl, va = evaluate(model, val_loader, loss_fn)
    history["train_loss"].append(tl); history["train_acc"].append(ta)
    history["val_loss"].append(vl); history["val_acc"].append(va)
    history["lr"].append(optimizer.param_groups[0]["lr"])


def fit(model, optimizer, n_epochs, train_epoch_fn=train_one_epoch, scheduler=None, verbose=False):
    torch.manual_seed(1)                       # same order of the mini-batches for every run
    history = new_history()
    for epoch in range(n_epochs):
        train_epoch_fn(model, train_loader, loss_fn, optimizer)
        record(history, model, optimizer)
        if scheduler is not None:
            scheduler.step()
        if verbose:
            print(f"epoch {epoch + 1:2d}  train loss {history['train_loss'][-1]:.3f}  "
                  f"val loss {history['val_loss'][-1]:.3f}  val acc {history['val_acc'][-1]:.3f}")
    return history


def plot_history(history, title=""):
    e = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    axes[0].plot(e, history["train_loss"], label="train"); axes[0].plot(e, history["val_loss"], label="validation")
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend()
    axes[1].plot(e, history["train_acc"], label="train"); axes[1].plot(e, history["val_acc"], label="validation")
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend()
    fig.suptitle(title)
    plt.show()


def plot_compare(histories, key="val_loss"):
    plt.figure(figsize=(7, 3.5))
    for name, h in histories.items():
        plt.plot(range(1, len(h[key]) + 1), h[key], label=name)
    plt.xlabel("epoch"); plt.ylabel(key); plt.legend()
    plt.show()


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())